# Multi-Model Arena with Iterative Refinement

**By: Dario Melconian**

### Agentic Design Patterns Used:
1. **Parallelization** — Multiple LLMs answer the same question simultaneously
2. **LLM-as-Judge (Evaluation)** — Claude evaluates competitor responses
3. **Iterative Refinement / Reflection** *(new!)* — Competitors receive Claude's critique and revise their answers. Claude then re-evaluates the improved responses.

### Model Setup:
- 🏆 **Judge**: `claude-sonnet-4-5` — strongest reasoner, handles evaluation + feedback
- **Competitor 1**: `gpt-4o-mini` (OpenAI)
- **Competitor 2**: Ollama local model (free, runs on your machine)
- **Competitor 3**: Groq — free API, ultra-fast Llama inference

### Finance Domain Twist:
The question generator is prompted to produce a **wealth management / financial reasoning** question — relevant to the real-world domain where evaluation quality and nuanced judgment matter most.

In [1]:
import os
import json
from dotenv import load_dotenv
from openai import OpenAI
from anthropic import Anthropic
from IPython.display import Markdown, display

load_dotenv(override=True)

ModuleNotFoundError: No module named 'dotenv'

In [ ]:
# --- API Key checks ---
openai_api_key = os.getenv('OPENAI_API_KEY')
anthropic_api_key = os.getenv('ANTHROPIC_API_KEY')
groq_api_key = os.getenv('GROQ_API_KEY')

print(f"OpenAI:    {'✅ ' + openai_api_key[:8] if openai_api_key else 'NOT SET'}")
print(f"Anthropic: {'✅ ' + anthropic_api_key[:7] if anthropic_api_key else 'NOT SET'}")
print(f"Groq:      {'✅ SET' if groq_api_key else 'NOT SET — get a free key at console.groq.com'}")

In [ ]:
# --- Clients ---
openai_client = OpenAI()
claude_client = Anthropic()

# Groq uses OpenAI-compatible API
groq_client = OpenAI(
    api_key=groq_api_key,
    base_url="https://api.groq.com/openai/v1"
)

# Ollama uses OpenAI-compatible API locally
ollama_client = OpenAI(
    api_key="ollama",
    base_url="http://localhost:11434/v1"
)

# --- Model names ---
JUDGE_MODEL = "claude-sonnet-4-5"           # Claude as judge
GPT_MODEL = "gpt-4o-mini"                   # Competitor 1
OLLAMA_MODEL = "llama3.2"                   # Competitor 2 
GROQ_MODEL = "llama-3.3-70b-versatile"      # Competitor 3 

competitors = ["GPT-4o-mini", f"Ollama ({OLLAMA_MODEL})", f"Groq ({GROQ_MODEL})"]

## Step 1 — Generate a Finance/Wealth Management Question

GPT-4o-mini generates a challenging, domain-specific question. We bias toward wealth management to make this relevant to real-world financial reasoning.

In [ ]:
request = (
    "Please come up with a challenging, nuanced question about wealth management, "
    "portfolio strategy, or financial planning that requires both quantitative reasoning "
    "and ethical judgment to answer well. "
    "Answer only with the question, no explanation."
)

response = openai_client.chat.completions.create(
    model=GPT_MODEL,
    messages=[{"role": "user", "content": request}]
)
question = response.choices[0].message.content
print("📋 Question generated:")
print(question)

## Step 2 — Round 1: All Competitors Answer

Parallelization pattern: same question sent to all three models.

In [ ]:
def get_openai_answer(question):
    response = openai_client.chat.completions.create(
        model=GPT_MODEL,
        messages=[{"role": "user", "content": question}]
    )
    return response.choices[0].message.content

def get_ollama_answer(question):
    response = ollama_client.chat.completions.create(
        model=OLLAMA_MODEL,
        messages=[{"role": "user", "content": question}]
    )
    return response.choices[0].message.content

def get_groq_answer(question):
    response = groq_client.chat.completions.create(
        model=GROQ_MODEL,
        messages=[{"role": "user", "content": question}]
    )
    return response.choices[0].message.content

print("⏳ Collecting Round 1 answers...")
answers_r1 = [
    get_openai_answer(question),
    get_ollama_answer(question),
    get_groq_answer(question),
]
print("✅ Round 1 complete.")

In [ ]:
# Display Round 1 answers
for competitor, answer in zip(competitors, answers_r1):
    display(Markdown(f"### 🤖 {competitor}\n\n{answer}\n\n---"))

## Step 3 — Claude Judges Round 1 and Provides Written Feedback

LLM-as-Judge pattern: Claude doesn't just rank — it gives specific, actionable critique to each competitor.

In [ ]:
def build_together(answers):
    together = ""
    for i, answer in enumerate(answers):
        together += f"# Response from Competitor {i+1} ({competitors[i]})\n\n{answer}\n\n"
    return together

together_r1 = build_together(answers_r1)

critique_prompt = f"""You are an expert judge evaluating responses to a wealth management question.

The question asked was:
{question}

Here are the responses from {len(competitors)} competitors:

{together_r1}

For each competitor, provide:
1. A brief critique (2-3 sentences) identifying what they did well and what was weak or missing
2. One specific suggestion they should act on in their revised answer

Format your response as JSON with this exact structure:
{{
  "feedback": [
    {{"competitor": 1, "critique": "...", "suggestion": "..."}},
    {{"competitor": 2, "critique": "...", "suggestion": "..."}},
    {{"competitor": 3, "critique": "...", "suggestion": "..."}}
  ]
}}

Respond with JSON only. No markdown formatting or code blocks."""

critique_response = claude_client.messages.create(
    model=JUDGE_MODEL,
    max_tokens=1500,
    messages=[{"role": "user", "content": critique_prompt}]
)

critique_raw = critique_response.content[0].text
critique_data = json.loads(critique_raw)
feedbacks = critique_data["feedback"]

print("📝 Claude's Round 1 Feedback:\n")
for fb in feedbacks:
    print(f"Competitor {fb['competitor']} ({competitors[fb['competitor']-1]}):")
    print(f"  Critique:   {fb['critique']}")
    print(f"  Suggestion: {fb['suggestion']}\n")

## Step 4 — Round 2: Competitors Revise Using Claude's Feedback

**Iterative Refinement / Reflection pattern** *(new agentic pattern!)*

Each model receives Claude's critique of their Round 1 answer and is asked to produce an improved response. This mirrors how a research analyst would revise a report after peer review.

In [ ]:
def build_refinement_prompt(question, original_answer, feedback):
    return (
        f"You previously answered this question:\n\n"
        f"{question}\n\n"
        f"Your original answer was:\n\n{original_answer}\n\n"
        f"An expert reviewer gave you this feedback:\n"
        f"Critique: {feedback['critique']}\n"
        f"Suggestion: {feedback['suggestion']}\n\n"
        f"Please write an improved version of your answer that addresses this feedback directly. "
        f"Be specific, clear, and thorough."
    )

print("⏳ Collecting Round 2 revised answers...")

answers_r2 = [
    get_openai_answer(build_refinement_prompt(question, answers_r1[0], feedbacks[0])),
    get_ollama_answer(build_refinement_prompt(question, answers_r1[1], feedbacks[1])),
    get_groq_answer(build_refinement_prompt(question, answers_r1[2], feedbacks[2])),
]

print("✅ Round 2 complete.")

In [ ]:
# Display Round 2 revised answers
for competitor, answer in zip(competitors, answers_r2):
    display(Markdown(f"### 🔄 {competitor} (Revised)\n\n{answer}\n\n---"))

## Step 5 — Claude's Final Judgment on Revised Answers

Claude evaluates the improved Round 2 responses and produces a final ranking with reasoning.

In [ ]:
together_r2 = build_together(answers_r2)

final_judge_prompt = f"""You are an expert judge in a final evaluation round.

Competitors were asked:
{question}

They each received feedback on their first attempt and have now submitted revised answers.
Evaluate their revised responses on:
- Depth and accuracy of financial reasoning
- Clarity and structure
- How well they incorporated the feedback

Revised responses:
{together_r2}

Respond with JSON only in this format:
{{
  "results": ["best competitor number", "second best", "third best"],
  "reasoning": "2-3 sentences explaining your final ranking"
}}

No markdown formatting or code blocks."""

final_response = claude_client.messages.create(
    model=JUDGE_MODEL,
    max_tokens=800,
    messages=[{"role": "user", "content": final_judge_prompt}]
)

final_raw = final_response.content[0].text
final_data = json.loads(final_raw)
ranks = final_data["results"]
reasoning = final_data["reasoning"]

print("🏆 FINAL RESULTS (after refinement)\n")
for index, result in enumerate(ranks):
    competitor = competitors[int(result) - 1]
    medal = ["🥇", "🥈", "🥉"][index]
    print(f"{medal} Rank {index+1}: {competitor}")

print(f"\n📊 Judge's reasoning: {reasoning}")

## Summary of Agentic Patterns Used

| Pattern | Where Used |
|---|---|
| **Parallelization** | Round 1 and Round 2 — same prompt sent to all competitors simultaneously |
| **LLM-as-Judge** | Claude evaluates responses in both rounds |
| **Iterative Refinement / Reflection** *(new!)* | Competitors receive written critique and revise their answers before final judgment |

### Commercial Relevance
This pattern mirrors real workflows in **wealth management research**: a junior analyst produces a draft, a senior reviewer provides structured feedback, and the analyst submits a revised version for final evaluation. Automating this loop with LLMs can dramatically improve output quality with minimal human intervention — valuable in any high-stakes, accuracy-critical domain.